# RecipeCatalog example

Shows how a catalog combines curated recipe stores and generated auto recipes behind one lookup surface.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from woodpecker.recipes import DatasetMatcher, FixRef, Recipe
from woodpecker.stores import AutoRecipeStore, JsonRecipeStore, RecipeCatalog
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset and a small curated recipe store.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)

tmp = TemporaryDirectory()
store = JsonRecipeStore(Path(tmp.name) / "recipes.yaml")
store.save_recipe(
    Recipe(
        id="cmip6.curated_units",
        description="Curated CMIP6 units recipe",
        match=DatasetMatcher(dataset_id_patterns=["CMIP6.CMIP.*.Amon.tas.*"]),
        steps=[FixRef(id="woodpecker.normalize_tas_units_to_kelvin")],
    )
)

catalog = RecipeCatalog([store, AutoRecipeStore()])

The catalog lists recipes from all sources and deduplicates by recipe id.

In [ ]:
[recipe.id for recipe in catalog.list_recipes()[:8]]

Lookup returns the curated recipe first, followed by the generated auto recipe.

In [ ]:
matched_plans = catalog.lookup(dataset)
[recipe.id for recipe in matched_plans]

The same catalog resolves recipe ids and aliases.

In [ ]:
catalog.get_recipe("cmip6.curated_units")

In [ ]:
tmp.cleanup()